# Real video: Qwen2.5-VL on NExT-QA frames (n=240, per-type breakdown)

End-to-end **video** measurement with the vmd harness: real frames from NExT-QA clips, Qwen2.5-VL-3B, and frame-sequence corruptions. Produces:

- **Collapse gap** — frames vs. no frames (language prior).
- **Temporal-order effect** — frames shuffled (content preserved, order destroyed).
- **Content effect** — frames dropped.
- **Per-type breakdown** — causal (C) / temporal (T) / descriptive (D) questions: ¿dónde duele el shuffle?

**Runtime**: ~6 configs × 240 items ≈ 1440 llamadas de inferencia. ~50–70 min en L4, menos en A100. Cada configuración se guarda al completarse (checkpoint): si se corta, re-ejecuta la celda del grid y retoma.

> ⚠️ Celdas en orden, idempotentes. Si algo queda raro: *Entorno de ejecución → Reiniciar*.

In [ ]:
# Setup idempotente
import os, shutil, sys
REPO = "/content/video-modality-diagnostics"
os.chdir("/content")
shutil.rmtree(REPO, ignore_errors=True)
!git clone -q https://github.com/mlahozy21/video-modality-diagnostics.git
os.chdir(REPO)
!pip install -q -e ".[video]"
for k in list(sys.modules):
    if k.startswith("vmd"):
        del sys.modules[k]
sys.path.insert(0, f"{REPO}/src")
import torch
print("setup OK —", os.getcwd(), "| CUDA:", torch.cuda.is_available())

In [ ]:
# Subconjunto real de NExT-QA (30 preguntas por cada uno de los 8 tipos)
N = 240
!python scripts/prepare_nextqa.py --n {N} --out-dir data/nextqa

In [ ]:
import json, os

from vmd.data import load_jsonl
from vmd.modalities import make_view
from vmd.video import VLMVideoBackend

items = load_jsonl("data/nextqa/videoqa.jsonl")
print(f"{len(items)} items de NExT-QA con vídeo real")

def qtype(item):
    return item.id.split("_")[1]          # CW/CH/TN/TC/TP/DC/DL/DO

def qgroup(item):
    return qtype(item)[0]                 # C / T / D

MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"

# configuracion -> (canales, severidad, tipo de corrupcion)
CONFIGS = {
    "full":        (["vision"], 0.0,  "shuffle"),
    "blind":       ([],         0.0,  "shuffle"),
    "shuffle_0.5": (["vision"], 0.5,  "shuffle"),
    "shuffle_1.0": (["vision"], 1.0,  "shuffle"),
    "drop_0.5":    (["vision"], 0.5,  "drop"),
    "drop_0.75":   (["vision"], 0.75, "drop"),
}

## Grid de evaluación (con checkpoint por configuración)

In [ ]:
CKPT = "records_video.json"
records = json.load(open(CKPT)) if os.path.exists(CKPT) else {}

backend = VLMVideoBackend(MODEL, n_frames=8)

for cfg, (mods, sev, kind) in CONFIGS.items():
    if cfg in records:
        print(f"{cfg}: ya hecho ({len(records[cfg])} items), salto")
        continue
    backend.corruption_kind = kind
    preds = {}
    for j, it in enumerate(items):
        view = make_view(it, mods, corrupt={"vision": sev} if sev else None,
                         kind="shuffle", seed=0)   # corrupcion textual no aplica (evidence vacia)
        preds[it.id] = int(backend.answer(it, view) == it.answer_idx)
        if (j + 1) % 60 == 0:
            print(f"  {cfg}: {j+1}/{len(items)}")
    records[cfg] = preds
    json.dump(records, open(CKPT, "w"))
    acc = sum(preds.values()) / len(preds)
    print(f"{cfg}: accuracy {acc:.3f}  [checkpoint guardado]")

print("\ngrid completo:", list(records))

## Tablas para el README

In [ ]:
import numpy as np

def acc(cfg, group=None):
    sel = [it for it in items if group is None or qgroup(it) == group]
    vals = [records[cfg][it.id] for it in sel if it.id in records[cfg]]
    return float(np.mean(vals)), len(vals)

print(f"| Metric (Qwen2.5-VL-3B, NExT-QA n={len(items)}) | Accuracy |")
print("|---|---:|")
labels = {"full": "Full accuracy (8 frames)", "blind": "Blind (no frames)",
          "shuffle_0.5": "Shuffle 50% of frames", "shuffle_1.0": "Shuffle 100% of frames",
          "drop_0.5": "Drop 50% of frames", "drop_0.75": "Drop 75% of frames"}
for cfg, lab in labels.items():
    print(f"| {lab} | {acc(cfg)[0]:.3f} |")
print(f"| **Collapse gap** | **{acc('full')[0] - acc('blind')[0]:.3f}** |")

print("\nPer question group (C=causal, T=temporal, D=descriptive):\n")
print("| Group | n | Full | Blind | Gap | Shuffle 100% | Δ shuffle | Drop 75% | Δ drop |")
print("|---|---:|---:|---:|---:|---:|---:|---:|---:|")
for g, name in [("C", "Causal"), ("T", "Temporal"), ("D", "Descriptive")]:
    full, n = acc("full", g)
    blind = acc("blind", g)[0]
    sh = acc("shuffle_1.0", g)[0]
    dr = acc("drop_0.75", g)[0]
    print(f"| {name} | {n} | {full:.3f} | {blind:.3f} | {full-blind:+.3f} "
          f"| {sh:.3f} | {sh-full:+.3f} | {dr:.3f} | {dr-full:+.3f} |")

## Cómo leerlo

- **Gap por grupo**: ¿en qué tipo de pregunta mira más el vídeo?
- **Δ shuffle en temporales**: si el modelo usara el orden, el shuffle debería doler *sobre todo* en T. Si Δ ≈ 0 también ahí, el sesgo bag-of-frames es fuerte incluso donde el orden es la pregunta.
- **Δ drop**: cuánto contenido necesita por grupo.

Descarga `records_video.json` antes de cerrar (panel Archivos) — contiene la predicción por item y permite cualquier análisis posterior sin re-inferencia.